In [1]:
import statsmodels.api as sm
import pandas as pd
import numpy as np
import yfinance as yf

In [2]:
univ = ['SPY','META']
px = yf.download(univ, start='2016-01-01')['Close']
px

[*********************100%***********************]  2 of 2 completed


Ticker,META,SPY
Date,,
2016-01-04,101.330154,169.471558
2016-01-05,101.835739,169.758163
2016-01-06,102.073639,167.616837
2016-01-07,97.067596,163.595428
2016-01-08,96.482727,161.799698
...,...,...
2026-06-12,566.454956,739.843994
2026-06-15,593.479980,752.890381
2026-06-16,600.210022,748.401978


In [3]:
## get returns. Mostly daily percent returns
ret = px / px.shift() - 1
ret

Ticker,META,SPY
Date,,
2016-01-04,NaN,NaN
2016-01-05,0.004989,0.001691
2016-01-06,0.002336,-0.012614
2016-01-07,-0.049043,-0.023992
2016-01-08,-0.006025,-0.010977
...,...,...
2026-06-12,-0.002551,0.005408
2026-06-15,0.047709,0.017634
2026-06-16,0.011340,-0.005962


In [4]:
## set X and Y. Have to add constant
X = ret[['SPY']]
X = sm.add_constant(X)

Y = ret['META']

In [5]:
print(X)
print(Y)

            const       SPY
Date                       
2016-01-04    1.0       NaN
2016-01-05    1.0  0.001691
2016-01-06    1.0 -0.012614
2016-01-07    1.0 -0.023992
2016-01-08    1.0 -0.010977
...           ...       ...
2026-06-12    1.0  0.005408
2026-06-15    1.0  0.017634
2026-06-16    1.0 -0.005962
2026-06-17    1.0 -0.012488
2026-06-18    1.0  0.010397

[2630 rows x 2 columns]
Date
2016-01-04         NaN
2016-01-05    0.004989
2016-01-06    0.002336
2016-01-07   -0.049043
2016-01-08   -0.006025
                ...   
2026-06-12   -0.002551
2026-06-15    0.047709
2026-06-16    0.011340
2026-06-17   -0.054364
2026-06-18    0.016984
Name: META, Length: 2630, dtype: float64


In [6]:
### can't do this - missing data

results = sm.OLS(Y, X).fit()

MissingDataError: exog contains inf or nans

In [9]:
X = X.dropna()

In [10]:
Y = Y.dropna()

In [11]:
results = sm.OLS(Y, X).fit()

In [13]:
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   META   R-squared:                       0.373
Model:                            OLS   Adj. R-squared:                  0.372
Method:                 Least Squares   F-statistic:                     1560.
Date:                Sat, 20 Jun 2026   Prob (F-statistic):          2.80e-268
Time:                        17:30:11   Log-Likelihood:                 6652.7
No. Observations:                2629   AIC:                        -1.330e+04
Df Residuals:                    2627   BIC:                        -1.329e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0001      0.000      0.350      0.726      -0.001       0.001
SPY            1.3205      0.033     39.501      0.000       1.255       1.386
==============================================================================
Omnibus:                      863.784   Durbin-Watson:                   1.964
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           127627.696
Skew:                          -0.402   Prob(JB):                         0.00
Kurtosis:                      37.124   Cond. No.                         88.9
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [14]:
results.params

const    0.000132
SPY      1.320454
dtype: float64

In [15]:
results.tvalues

const     0.350185
SPY      39.501493
dtype: float64

In [16]:
## MEta reconstruction
META = results.params['SPY']*X['SPY'] + results.params['const'] + results.resid

In [17]:
(META-Y).abs().sum()

np.float64(1.8344700758454735e-16)

In [18]:
beta_constr = results.params['SPY']*X['SPY']
prediction = results.params['SPY']*X['SPY'] + results.params['const']
alpha_contr = results.params['const'] + results.resid

In [19]:
alpha_contr.corr(X['SPY'])

np.float64(1.0036466612824145e-15)

In [20]:
### Alpha contribution worked out
alpha_contr.mean()

np.float64(0.00013183188989168517)

In [21]:

### Information ratio is worked out
alpha_contr.mean()/alpha_contr.std()*np.sqrt(252)

np.float64(0.10860781579730845)

In [22]:
### alpha t - stat worked out
results.tvalues['const']

np.float64(0.35018529936261444)